# 03 - Analysis
Cross-correlation, Granger causality, event detection, and impact assessment.

In [1]:
import sys
sys.path.insert(0, "..")
import logging
logging.basicConfig(level=logging.INFO)
import pandas as pd
import numpy as np

## Step 1: Load data

In [2]:
import datetime
from pathlib import Path
import yaml
from src.polymarket.market_discovery import load_discovered_markets
from src.freight.scraper import fetch_all_freight_indexes
from src.freight.normalize import prepare_freight_panel

PANEL_PATH = Path('../data/processed/timeseries_panel.csv')

markets_df = load_discovered_markets()

# Read study-period start from settings
with open('../config/settings.yaml') as _f:
    _settings = yaml.safe_load(_f)
STUDY_START = _settings['analysis']['study_period']['start']

# ── Timeseries: load from panel CSV built in NB 02 ───────────────────────────
if PANEL_PATH.exists():
    panel = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    timeseries = {mid: grp.reset_index(drop=True) for mid, grp in panel.groupby('market_id')}
    print(f'Loaded timeseries panel from cache: {len(timeseries)} markets, {len(panel):,} rows')
else:
    # Fallback: fetch from API (run NB 02 first to avoid this)
    print(f'Panel CSV not found — fetching from API (study start: {STUDY_START}). Run NB 02 first to cache results.')
    from src.polymarket.timeseries import TimeseriesFetcher
    markets_filtered = markets_df[markets_df['category'].notna()].copy()
    markets_filtered['_created_dt'] = pd.to_datetime(
        markets_filtered['created_at'], errors='coerce', utc=True
    )
    markets_filtered = markets_filtered[
        markets_filtered['_created_dt'].isna() |
        (markets_filtered['_created_dt'] >= STUDY_START)
    ].drop(columns=['_created_dt'])
    study_start_ts = int(datetime.datetime.fromisoformat(STUDY_START).replace(
        tzinfo=datetime.timezone.utc).timestamp())
    fetcher = TimeseriesFetcher()
    timeseries = fetcher.fetch_all(markets_filtered, start_ts=study_start_ts, max_workers=8)
    fetcher.build_panel(timeseries)

# ── Freight data ─────────────────────────────────────────────────────────────
freight_raw = fetch_all_freight_indexes(use_synthetic_fallback=True)
freight_data = prepare_freight_panel(freight_raw)

print(f'Markets: {len(timeseries)}, Freight indexes: {list(freight_data.keys())}')

INFO:src.polymarket.timeseries:Fetching timeseries for 7119 markets …


Panel CSV not found — fetching from API (study start: 2023-01-01). Run NB 02 first to cache results.


INFO:src.polymarket.timeseries:5790 markets loaded from cache; 1329 require API calls.
INFO:src.polymarket.timeseries:Fetching history for market 508503 (token 21151499090639157023105520573782804187006240987910179362864625151010763665435) …
INFO:src.polymarket.timeseries:Fetching history for market 552064 (token 35165864485131867490876018829378241981100245790382503055069974672337051187204) …
INFO:src.polymarket.timeseries:Fetching history for market 1096589 (token 10001945247466016958890681330895387918419066882066596254381999848917758583889) …
INFO:src.polymarket.timeseries:Fetching history for market 554701 (token 39151492672543965221487457534020868730976246502230960123570360903147707612370) …
INFO:src.polymarket.timeseries:Fetching history for market 1096586 (token 12799381267496135147768698855810031078131966696677797752451254533525682123689) …
INFO:src.polymarket.timeseries:Fetching history for market 528600 (token 95758009727749070198233551272938108795936729777567095937625553412173

Markets: 5790, Freight indexes: ['BDI', 'FBX_GLOBAL', 'FBX01', 'FBX03', 'FBX11']


## Step 2: Event Detection

In [3]:
from src.analysis.events import EventDetector

detector = EventDetector()
all_events = detector.detect_all(timeseries, markets_df)
events_df = detector.to_dataframe(all_events)
print(f'Detected {len(all_events)} significant probability shift events')
events_df.head(20)

INFO:src.analysis.events:Event detection complete: 4075 events across 5790 markets.


Detected 4075 significant probability shift events


,market_id,market_title,timestamp,probability_before,probability_after,delta,direction,magnitude,detection_method,zscore
0,248427,Will the Philadelphia Eagles win Super Bowl LVII?,2023-01-30,0.16,0.54,0.38,up,0.38,threshold,NaN
1,248427,Will the Philadelphia Eagles win Super Bowl LVII?,2023-02-13,0.54,0.76,0.22,up,0.22,threshold,NaN
2,248427,Will the Philadelphia Eagles win Super Bowl LVII?,2023-02-22,0.76,0.50,-0.26,down,0.26,threshold,NaN
3,248861,NBA: Philadelphia 76ers vs. Memphis Grizzlies ...,2023-02-24,0.62,0.50,-0.12,down,0.12,zscore,-9.000669
4,248913,NBA: Philadelphia 76ers vs. Miami Heat 2023-02-27,2023-02-28,0.69,0.50,-0.19,down,0.19,zscore,-8.775686
5,248935,NBA: Miami Heat vs. Philadelphia 76ers 2023-03-01,2023-03-01,0.50,0.73,0.23,up,0.23,zscore,6.204837
6,248944,NBA: Dallas Mavericks vs. Philadelphia 76ers 2...,2023-03-02,0.50,0.57,0.07,up,0.07,zscore,6.204837
7,248982,NBA: Milwaukee Bucks vs. Philadelphia 76ers 20...,2023-03-04,0.50,0.65,0.15,up,0.15,zscore,6.082763
8,248999,NBA: Indiana Pacers vs. Philadelphia 76ers 202...,2023-03-06,0.50,0.27,-0.23,down,0.23,zscore,-6.000000
9,248935,NBA: Miami Heat vs. Philadelphia 76ers 2023-03-01,2023-03-08,0.73,0.50,-0.23,down,0.23,threshold,NaN


In [4]:
# Top events by magnitude
top_events = detector.get_top_events(all_events, n=10)
top_events_df = detector.to_dataframe(top_events)
print('Top 10 most significant probability shifts:')
top_events_df[['market_title', 'timestamp', 'probability_before',
               'probability_after', 'delta', 'magnitude', 'direction']]

Top 10 most significant probability shifts:


,market_title,timestamp,probability_before,probability_after,delta,magnitude,direction
0,Will Trump increase tariffs on Canada before May?,2025-05-01,0.0335,0.9995,0.9660,0.9660,up
1,Will Trump increase tariffs on Mexico before May?,2025-05-02,0.0205,0.9995,0.9790,0.9790,up
2,China-Vietnam trade deal before June?,2025-05-22,0.0290,0.9850,0.9560,0.9560,up
3,Will courts block Trump's tariffs before June?,2025-05-29,0.0345,0.9990,0.9645,0.9645,up
4,"Will ""Sinners"" win Best Compilation Soundtrack...",2026-02-01,0.0370,0.9960,0.9590,0.9590,up
5,Will Stephen Miran be the first to leave the T...,2026-02-03,0.0275,0.9995,0.9720,0.9720,up
6,Will the Pheu Thai Party (PT) win between 70 a...,2026-02-14,0.0120,0.9770,0.9650,0.9650,up
7,Will Zhongyan Ning (CHN) win the gold medal fo...,2026-02-19,0.0240,0.9995,0.9755,0.9755,up
8,Will Antionette Rijpma-De Jong (NED) win the g...,2026-02-20,0.0400,0.9995,0.9595,0.9595,up
9,Will Johannes Dale-Skjevdal (NOR) win the gold...,2026-02-20,0.0505,0.9995,0.9490,0.9490,up


## Step 3: Cross-Correlation Analysis

In [5]:
from src.analysis.correlation import CorrelationAnalyser

analyser = CorrelationAnalyser()
xcorr_results = analyser.run_cross_correlations(timeseries, freight_data, markets_df)
xcorr_df = analyser.xcorr_to_dataframe(xcorr_results)
print(f'Cross-correlation computed for {len(xcorr_results)} market-freight pairings')
xcorr_df.sort_values('peak_correlation', ascending=False)

INFO:src.analysis.correlation:Running cross-correlation for 13187 pairings …
INFO:src.analysis.correlation:  507300 × FBX03: peak r=-0.292 at lag=23 (significant)
INFO:src.analysis.correlation:  507300 × FBX_GLOBAL: peak r=0.212 at lag=-30 (significant)
INFO:src.analysis.correlation:  509105 × FBX03: peak r=-0.169 at lag=-18 (significant)
INFO:src.analysis.correlation:  509105 × FBX_GLOBAL: peak r=-0.258 at lag=1 (significant)
INFO:src.analysis.correlation:  1198423 × BDI: peak r=-0.643 at lag=18 (significant)
INFO:src.analysis.correlation:  532741 × BDI: peak r=-0.383 at lag=29 (significant)
INFO:src.analysis.correlation:  507284 × FBX03: peak r=0.228 at lag=8 (significant)
INFO:src.analysis.correlation:  507284 × FBX_GLOBAL: peak r=0.281 at lag=-27 (significant)
INFO:src.analysis.correlation:  572480 × BDI: peak r=-0.517 at lag=19 
INFO:src.analysis.correlation:  916732 × BDI: peak r=-0.820 at lag=-20 (significant)
INFO:src.analysis.correlation:  567030 × FBX11: peak r=-0.970 at lag=

Cross-correlation computed for 1102 market-freight pairings


,market_id,market_title,freight_index,peak_lag_days,peak_correlation,peak_p_value,is_significant,polymarket_leads,n_observations,interpretation
988,646823,Will Joey Aguilar win the 2025–2026 Davey O'Br...,FBX03,-23,1.000000,0.000000e+00,True,False,35,Peak correlation r=1.000 at lag=-23: freight L...
989,646823,Will Joey Aguilar win the 2025–2026 Davey O'Br...,FBX_GLOBAL,-23,1.000000,0.000000e+00,True,False,35,Peak correlation r=1.000 at lag=-23: freight L...
149,896775,Will China win the most gold medals in the 202...,FBX03,-19,0.999989,2.910275e-22,True,False,30,Peak correlation r=1.000 at lag=-19: freight L...
518,923369,Will Chaikasem Nitisiri be the next prime mini...,FBX03,-16,0.999989,2.709002e-29,True,False,30,Peak correlation r=1.000 at lag=-16: freight L...
810,1170164,Will Rosanna Mailan win Sing for Greece 2026?,FBX03,-24,0.999305,1.017614e-12,True,False,34,Peak correlation r=0.999 at lag=-24: freight L...
...,...,...,...,...,...,...,...,...,...,...
621,692253,Will China GDP growth in Q4 2025 be between 3....,FBX03,29,-0.994744,3.213641e-10,True,True,40,Peak correlation r=-0.995 at lag=29: Polymarke...
254,923366,Will Anutin Charnvirakul be the next prime min...,FBX03,-16,-0.994797,2.832190e-13,True,False,30,Peak correlation r=-0.995 at lag=-16: freight ...
939,250224,NBA: Philadelphia 76ers vs. Brooklyn Nets 2023...,FBX03,17,-0.999182,4.305360e-18,True,True,31,Peak correlation r=-0.999 at lag=17: Polymarke...
1060,250151,NBA: Philadelphia 76ers vs. Brooklyn Nets 2023...,FBX03,19,-0.999756,3.075330e-21,True,True,33,Peak correlation r=-1.000 at lag=19: Polymarke...


In [6]:
# Show significant results
sig_df = xcorr_df[xcorr_df['is_significant']]
print(f'Statistically significant pairings (p < 0.05): {len(sig_df)}')
print(f"Pairings where Polymarket leads freight: {sig_df['polymarket_leads'].sum()}")
sig_df[['market_title', 'freight_index', 'peak_lag_days',
        'peak_correlation', 'peak_p_value', 'interpretation']]

Statistically significant pairings (p < 0.05): 1021
Pairings where Polymarket leads freight: 551


,market_title,freight_index,peak_lag_days,peak_correlation,peak_p_value,interpretation
0,Will Inter Milan win the UEFA Champions League?,FBX03,23,-0.291840,0.000006,Peak correlation r=-0.292 at lag=23: Polymarke...
1,Will Inter Milan win the UEFA Champions League?,FBX_GLOBAL,-30,0.212138,0.001336,Peak correlation r=0.212 at lag=-30: freight L...
2,Will the Philadelphia Flyers win the 2025 Stan...,FBX03,-18,-0.169328,0.031768,Peak correlation r=-0.169 at lag=-18: freight ...
3,Will the Philadelphia Flyers win the 2025 Stan...,FBX_GLOBAL,1,-0.257749,0.000514,Peak correlation r=-0.258 at lag=1: Polymarket...
4,"US strikes Iran by February 28, 2026?",BDI,18,-0.643355,0.024004,Peak correlation r=-0.643 at lag=18: Polymarke...
...,...,...,...,...,...,...
1097,MLB: New York Yankees vs. Philadelphia Phillie...,FBX_GLOBAL,11,0.699095,0.000003,Peak correlation r=0.699 at lag=11: Polymarket...
1098,MLB: Chicago White Sox vs. Philadelphia Philli...,FBX03,18,-0.734441,0.004250,Peak correlation r=-0.734 at lag=18: Polymarke...
1099,MLB: Chicago White Sox vs. Philadelphia Philli...,FBX_GLOBAL,18,-0.616595,0.024801,Peak correlation r=-0.617 at lag=18: Polymarke...
1100,Will the Philadelphia Flyers be Eastern Confer...,FBX03,16,0.762107,0.000059,Peak correlation r=0.762 at lag=16: Polymarket...


## Step 4: Granger Causality

In [7]:
granger_results = analyser.run_granger_tests(timeseries, freight_data, markets_df)
granger_df = analyser.granger_to_dataframe(granger_results)
print(f'Granger tests run for {len(granger_results)} pairings')
print(f"Significant (p < 0.05): {granger_df['is_significant'].sum()}")
granger_df.sort_values('min_p_value')

# ── Persist analysis results for NB 04 ───────────────────────────────────────
import pickle
ANALYSIS_CACHE = Path('../data/processed/analysis_cache.pkl')
with open(ANALYSIS_CACHE, 'wb') as f:
    pickle.dump({
        'all_events': all_events,
        'xcorr_results': xcorr_results,
        'granger_results': granger_results,
    }, f)
print(f'Analysis cache saved → {ANALYSIS_CACHE}')

INFO:src.analysis.correlation:Running Granger tests for 13187 pairings …
c:\Users\kekoi\Projects\operations_research\polymarket-scm-intel\venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
INFO:src.analysis.correlation:  507300 → FBX03: p=0.3942 (lag=2) 
c:\Users\kekoi\Projects\operations_research\polymarket-scm-intel\venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
INFO:src.analysis.correlation:  507300 → FBX_GLOBAL: p=0.3433 (lag=2) 
c:\Users\kekoi\Projects\operations_research\polymarket-scm-intel\venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
INFO:src.analysis.correlation:  509105 → FBX03: p=0.2646 (lag=1) 
c:\Users\kekoi\Projects\operations_research\polymarket-scm-intel\venv\

Granger tests run for 532 pairings
Significant (p < 0.05): 187
Analysis cache saved → ..\data\processed\analysis_cache.pkl


## Step 5: Impact Assessment

In [8]:
from src.analysis.impact_mapper import ImpactMapper

mapper = ImpactMapper()
assessments = mapper.generate_assessments(all_events, markets_df, xcorr_results)
assessments_df = mapper.to_dataframe(assessments)
print(f'Generated {len(assessments)} impact assessments')
assessments_df[['signal', 'timestamp', 'confidence', 'impact_score', 'category']].head(10)

INFO:src.analysis.impact_mapper:Generated 3218 impact assessments.


Generated 3218 impact assessments


,signal,timestamp,confidence,impact_score,category
0,Houthi strike on Israel by August 31?: probabi...,2025-08-28,high,0.9105,red_sea_houthi
1,Will the Supreme Court rule on Trump's tarriff...,2026-02-20,high,0.7997,us_trade_policy
2,Will Trump visit China by March 31?: probabili...,2026-02-21,high,0.7355,tariffs_us_china
3,Will Susan Crawford win by 10% or more?: proba...,2025-04-08,high,0.7347,us_trade_policy
4,China-Vietnam trade deal before June?: probabi...,2025-05-22,high,0.7049,tariffs_us_china
5,Will Trump agree to a tariff agreement with Me...,2025-08-03,high,0.6172,tariffs_us_china
6,Will courts block Trump's tariffs before June?...,2025-05-29,high,0.6090,tariffs_us_china
7,Will Saeed Jalili be the next President of Ira...,2024-07-07,high,0.5962,iran_hormuz
8,Another successful Houthi attack on shipping i...,2025-08-07,high,0.5621,red_sea_houthi
9,Set Handicap: Samsonova (-1.5) vs Siegemund (+...,2026-01-26,high,0.5481,labor_disruption


In [9]:
# Print the top 3 assessments in full
for a in assessments[:3]:
    print('=' * 70)
    print(f'SIGNAL: {a.signal}')
    print(f'Date: {a.timestamp} | Category: {a.category} | Confidence: {a.confidence.upper()}')
    print(f'Impact Score: {a.impact_score:.4f}')
    print(f'Affected Routes: {", ".join(a.affected_routes)}')
    print(f'Predicted Impact: {a.predicted_impact}')
    print(f'Expected Range: {a.predicted_impact_range}')
    print(f'Recommended Actions:')
    for action in a.recommended_actions[:4]:
        print(f'  - {action}')
    print(f'Historical Precedent: {a.historical_precedent}')
    print()

SIGNAL: Houthi strike on Israel by August 31?: probability rose from 6% to 99% (+93pp shift)
Date: 2025-08-28 | Category: red_sea_houthi | Confidence: HIGH
Impact Score: 0.9105
Affected Routes: FBX11: China/East Asia → North Europe, FBX_GLOBAL: Global Container Index
Predicted Impact: Elevated Houthi attack probability forces Suez Canal avoidance; Cape routing adds cost and time to Asia-Europe lanes
Expected Range: 20–40% FBX11 increase within 1–3 weeks as Suez rerouting accelerates
Recommended Actions:
  - Rebook Asia-Europe shipments via Cape of Good Hope routing (add 10–14 days)
  - Increase Europe-bound inventory lead times by 2–3 weeks
  - Secure space on alternative Cape-routed vessels immediately
  - Assess war risk surcharge exposure on current bookings
Historical Precedent: Jan 2024 Houthi escalation drove FBX11 from $2,200 to over $7,000 within 6 weeks

SIGNAL: Will the Supreme Court rule on Trump's tarriffs by February 20?: probability rose from 15% to 99% (+84pp shift)
Date

## Summary
Analysis complete. Proceed to 04_whitepaper_figures.ipynb for publication-quality charts.